# 3 - A partir das extremidades das águas correntes e das águas correntes estimadas, traçar um círculo de 50m;


In [1]:
import geopandas as gpd
import pandas as pd
import shapely
from os.path import join
from tqdm import tqdm


In [2]:
def testar_gdf(gdf):
    print(
    f'Shape: {gdf.shape};\n'+
    f'\nSample:{gdf.sample()}'
)

# Raw gdf

In [3]:
drenageo = gpd.read_file(
    join(
        'data',
        'drenagem.zip'
    )
)

# Silver gdf

In [4]:
## Como as correntes estimadas tem a id como int, vou transformar a id do drenageo em int tbm
drenageo['cd_identif'].astype('int', copy=False)
## Conferir se todos os 'cd_tipo_cu' sejam do mesmo tipo
drenageo['cd_tipo_cu'].dtype

dtype('float64')

# Determinar Correntes Estimadas (só por enquanto, dps vamos usar o do Elias e tals) 

In [5]:
drenageo.sample(10)
#* 11: trecho em estado natural
#* 12: lago ou reservatório
#* 10: trecho fechado
#* 9: trecho a céu aberto

cus_to_keep = [9.0, 11.0]
colors_dictionarie= {
    9.0 : 'turquoise',
    11.0 : 'aquamarine',
    10.0 : 'pink',
    12.0 : 'pink',
}

drenageo['colors'] = drenageo['cd_tipo_cu'].map(colors_dictionarie)

# Create GDF copy

In [6]:
gdf= drenageo[[
    'cd_identif', 
    'cd_tipo_ac', 
    'cd_tipo_cu', 
    'nm_acident', 
    'geometry'
]]

# Pontos buff

Pelo que eu vi e conversei com o Mauryas: ```Considere os tipos para efeito de continuidade. Só não gere a nascente se esses tipos desconhecidos forem os finais de um curso.```  
Sabe o que isso quer dizer também? Que não dá pra eu separar por nomes igual o Henrique fez...

Eu descobri pq o touches não dava certo e agora acho que tudo isso foi meio em vão actually... faz pensar, né?  
Esse tempo todo, eu precisava usar o `.intersec()`, veja:

In [7]:
pontos = gdf.copy()
pontos_0 = gdf.copy()
pontos_1 = gdf.copy()
pontos_buff = gdf.copy()

pontos_0['geometry']=shapely.get_point(gdf.geometry, 0)
pontos_0['cd_point'] = pontos_0['cd_identif'].astype(int).astype(str)+".0"
pontos_1['geometry'] = shapely.get_point(gdf.geometry, -1)
pontos_1['cd_point'] = pontos_1['cd_identif'].astype(int).astype(str)+".1"

Agora damos buffer nos pontos, para tirarmos os que intersectam com alguma coisa:

In [8]:
pontos = pd.concat([pontos_0, pontos_1], ignore_index=True)
pontos_buff['geometry'] = pontos['geometry'].buffer(10)

In [9]:
pontos_buff.sample()

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry
23940,22859.0,ND,11.0,SD,"POLYGON ((323466.362 7367433.34, 323466.313 73..."


In [10]:
#E vamos, claro, conferir as intersecções
intersecs_bool=[]

for i, row in pontos_buff.iterrows():
    outras_geoms = pontos_buff.loc[pontos_buff.index!=i]
    outras_geoms.sample()
    intersecs_bool = row.geometry.intersects(outras_geoms.geometry)
    if len(intersecs_bool.loc[intersecs_bool==True])<1:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool'] = (
            False
        )
    else:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool'] = (
            True
        )

In [11]:
for i, row in pontos_buff.loc[pontos_buff['intersec_bool']==False].iterrows():
    outras_linhas = gdf.loc[gdf['cd_identif']!=row['cd_identif']]
    outras_linhas
    intersecs_bool = row.geometry.intersects(outras_linhas.geometry)
    if len(intersecs_bool.loc[intersecs_bool==True])<1:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool']= False
    else:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool'] = True

In [12]:
pontos_buff.shape

(27611, 6)

In [13]:
pontos_buff['cd_tipo_cu'].astype(dtype='float', copy=False)
pontos_buff= pontos_buff.loc[pontos_buff['cd_tipo_cu'].isin(cus_to_keep)]
pontos_buff = pontos_buff.loc[pontos_buff['intersec_bool']==False]

In [14]:
pontos_buff.shape

(8279, 6)

In [15]:
pontos_buff.sample(3)

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry,intersec_bool
2394,2245.0,ND,11.0,SD,"POLYGON ((331510.295 7359461.814, 331510.247 7...",False
26382,26872.0,ND,11.0,SD,"POLYGON ((332224.695 7406204.592, 332224.647 7...",False
3740,3530.0,ND,11.0,SD,"POLYGON ((320247.965 7345093.434, 320247.917 7...",False


In [16]:
drenageo.sample()

,cd_identif,cd_tipo_ac,tx_tipo_ac,cd_numero_,nm_bairro,nm_acident,qt_comprim,cd_tipo_cu,nm_tipo_cu,nm_via_pro,nm_descrit,nm_tipo_tr,dt_atualiz,cd_usuario,geometry,colors
19899,18953.0,ND,None,2,S/B,SD,21.883184,11.0,Trecho em estado natural,S/N,None,Trecho a céu aberto,2025-01-03,None,"LINESTRING (319950.55 7360212.182, 319942.821 ...",aquamarine


# Visualizar
### Ok, eu estava errada... mesmo com o intersec e um mega buffer nos pontos, ainda não dá certo, vamos voltar pra tatica do Henrique mesmo
m= drenageo.explore(color='pink')
drenageo.loc[drenageo['nm_acident']=="CORREGO MANDAQUI"].explore(m=m, color='red')
pontos_buff.explore(
    m=m,
    color="green"
)

In [17]:
# Salvar arquivo para a validação do Mauryas
pontos_buff.to_file(
    join(
        'data',
        'pontos_buff_corrigido.geojson'
    ),
    driver="GeoJSON"
)